# Stage 2 — Where Is It Going Wrong?

Stage 1 told you *whether* the chatbot gets answers right. When it doesn't, Stage 2 tells
you **which part of the pipeline to fix**: retrieval or generation.

The only new requirement over Stage 1 is that you **capture the retrieved contexts**
alongside each answer — the chunks your RAG pipeline handed to the LLM. Most RAG
frameworks (LangChain, LlamaIndex, Llama Stack, ...) expose these directly in the
response object.

**What we measure** (0–1, higher is better):

| Metric | Question it answers | Low score means |
|--------|--------------------|----------------|
| `faithfulness` | Does the answer stick to what was retrieved? | **Generation problem** — the model hallucinates beyond its sources |
| `context_precision` | Are the retrieved chunks actually relevant? | **Noisy retrieval** — the model wades through junk to find the signal |
| `context_recall` | Did retrieval surface everything needed to answer? | **Retrieval problem** — the right documents never reached the model |
| `answer_relevancy` | Does the answer actually address the question? | Evasive or off-topic answers |

> **Note on this example:** because this guide doesn't have access to your chatbot's
> internals, the cells below *simulate* a small RAG pipeline (a mini document corpus +
> embedding-based retrieval + an LLM). In your environment, delete the simulation and
> populate `retrieved_contexts` with the chunks your real pipeline returns.

> **Prerequisites:** same as Stage 1 — see [`README.md`](README.md).


In [ ]:
%pip install -q "eval-hub-sdk[client]" boto3 openai pandas numpy requests python-dotenv


## Configuration

Identical to Stage 1.


In [ ]:
import os
import subprocess

from dotenv import load_dotenv

load_dotenv()


def _oc(*args: str) -> str:
    return subprocess.run(["oc", *args], capture_output=True, text=True).stdout.strip()


# --- EvalHub service ---
EVALHUB_URL = os.environ.get("EVALHUB_URL") or "https://" + _oc(
    "get", "route", "evalhub", "-n", "evalhub", "-o", "jsonpath={.spec.host}"
)
EVALHUB_TENANT = os.environ.get("EVALHUB_TENANT", "evalhub")
EVALHUB_TOKEN = os.environ.get("EVALHUB_TOKEN") or _oc("whoami", "-t")

# --- The chatbot (LLM side of the simulated RAG pipeline) ---
CHATBOT_URL = os.environ.get("LITELLM_API_URL", "https://your-chatbot.example.com/v1")
CHATBOT_API_KEY = os.environ.get("LITELLM_API_KEY", "")
CHATBOT_MODEL = os.environ.get("LITELLM_MODEL", "llama-scout-17b")

# --- Judge model used by RAGAS ---
JUDGE_URL = os.environ.get("JUDGE_URL", CHATBOT_URL)
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", CHATBOT_MODEL)
JUDGE_EMBEDDING_MODEL = os.environ.get("JUDGE_EMBEDDING_MODEL", "nomic-embed-text-v1-5")
JUDGE_AUTH_SECRET = "judge-model-auth"

# --- S3 storage for the dataset ---
S3_ENDPOINT = os.environ.get("S3_ENDPOINT") or "https://" + _oc(
    "get", "route", "minio", "-n", "evalhub", "-o", "jsonpath={.spec.host}"
)
S3_ACCESS_KEY = os.environ.get("S3_ACCESS_KEY", "minio")
S3_SECRET_KEY = os.environ.get("S3_SECRET_KEY", "minio12345")
S3_BUCKET = os.environ.get("S3_BUCKET", "evals")
S3_SECRET_REF = "evalhub-test-data-s3"  # k8s secret the job uses to download the data

# --- MLflow experiment tracking ---
# All runs (stage 1 and stage 2) land in one experiment so you can compare them.
MLFLOW_URL = os.environ.get("MLFLOW_URL") or "https://" + _oc(
    "get", "route", "mlflow", "-n", "evalhub", "-o", "jsonpath={.spec.host}"
)
EXPERIMENT_NAME = os.environ.get("MLFLOW_EXPERIMENT", "rag-chatbot-eval")

print(f"EvalHub: {EVALHUB_URL} (tenant: {EVALHUB_TENANT})")
print(f"Chatbot: {CHATBOT_MODEL} @ {CHATBOT_URL}")
print(f"Judge:   {JUDGE_MODEL} @ {JUDGE_URL}")


## 1. Test set and document corpus

Same idea as Stage 1 — questions plus reference answers — but now we also have the
document corpus the RAG pipeline retrieves from. In your environment this is your real
knowledge base; here it's a handful of inline passages (plus some distractors, so
retrieval can actually make mistakes).


In [ ]:
test_set = [
    {
        "question": "What does the DOP label mean on Italian food products?",
        "reference": "DOP (Denominazione di Origine Protetta, Protected Designation of Origin) certifies that a product is produced, processed, and prepared entirely in a specific geographic area, using recognized local know-how.",
    },
    {
        "question": "What is the difference between DOP and IGP certifications?",
        "reference": "DOP requires that all production phases happen in the designated area, while IGP only requires that at least one production phase takes place in the area.",
    },
    {
        "question": "Which region of Italy produces Parmigiano Reggiano?",
        "reference": "Parmigiano Reggiano is produced in the provinces of Parma, Reggio Emilia, Modena, and parts of Bologna and Mantua.",
    },
    {
        "question": "What conditions must a product meet to be labelled organic in the EU?",
        "reference": "EU organic products must be grown without synthetic pesticides and fertilizers, without GMOs, respecting animal welfare standards, and be certified by an approved control body under EU Regulation 2018/848.",
    },
    {
        "question": "Why are bees important for agriculture?",
        "reference": "Bees pollinate a large share of food crops, including fruits, vegetables, and seed crops; without pollinators, yields and quality of many crops would drop sharply.",
    },
    {
        "question": "How does drip irrigation save water?",
        "reference": "Drip irrigation delivers water directly to the plant root zone through emitters, reducing evaporation and runoff losses compared to wetting the whole field surface.",
    },
]

corpus = [
    # Relevant documents
    "DOP stands for Denominazione di Origine Protetta (Protected Designation of Origin). The label certifies that a food product is produced, processed, and prepared entirely within a specific geographic area, following a strict production protocol that draws on recognized local know-how.",
    "IGP (Indicazione Geografica Protetta) requires that at least one phase of production takes place in the designated geographic area. This is less strict than DOP, where every production phase must occur in the area.",
    "Parmigiano Reggiano can only be produced in a delimited area of northern Italy covering the provinces of Parma, Reggio Emilia, Modena, and parts of the provinces of Bologna and Mantua.",
    "Under EU Regulation 2018/848, organic products must be grown without synthetic pesticides or fertilizers and without GMOs, must respect animal welfare standards, and must be certified by an approved control body before carrying the EU organic logo.",
    "Bees and other pollinators are responsible for pollinating a large share of the crops humans eat, including most fruits, vegetables, and seed crops. A decline in pollinator populations directly threatens crop yields, food quality, and biodiversity.",
    "Drip irrigation systems deliver water directly to the root zone of each plant through a network of pipes and emitters. Because the water is not sprayed over the whole field, losses from evaporation and runoff are greatly reduced.",
    # Distractors
    "The Chianti wine region in Tuscany is known for red wines made primarily from Sangiovese grapes. Chianti Classico carries the historic Gallo Nero (black rooster) seal.",
    "Olive harvesting in the Mediterranean traditionally takes place between October and December, with early harvests producing more peppery, polyphenol-rich oils.",
    "The European Green Deal aims to make the EU climate neutral by 2050, with the Farm to Fork strategy targeting a reduction in pesticide use and an increase in organic farming area.",
    "Vertical farming grows crops in stacked indoor layers under LED lighting, using hydroponic or aeroponic systems that recycle water and require no arable land.",
]

print(f"{len(test_set)} questions, {len(corpus)} documents in the corpus")


## 2. The (simulated) RAG pipeline

A minimal but realistic RAG loop: embed the corpus, retrieve the top-k most similar
passages for each question, and have the LLM answer **using only those passages**.

**In your environment, replace this whole section** with calls to your chatbot, keeping
the contexts it retrieved for each question.


In [ ]:
import numpy as np
from openai import OpenAI

llm = OpenAI(base_url=CHATBOT_URL, api_key=CHATBOT_API_KEY)


def embed(texts: list[str]) -> np.ndarray:
    resp = llm.embeddings.create(model=JUDGE_EMBEDDING_MODEL, input=texts)
    vectors = np.array([d.embedding for d in resp.data])
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


corpus_embeddings = embed(corpus)


def retrieve(question: str, top_k: int = 2) -> list[str]:
    """Return the top_k most similar corpus passages for the question."""
    q = embed([question])[0]
    scores = corpus_embeddings @ q
    return [corpus[i] for i in np.argsort(scores)[::-1][:top_k]]


RAG_PROMPT = (
    "Answer the question using ONLY the provided context. "
    "If the context does not contain the answer, say so. "
    "Answer concisely in 2-3 sentences.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)


def rag_chatbot(question: str) -> tuple[str, list[str]]:
    """Simulated RAG chatbot: returns (answer, retrieved_contexts)."""
    contexts = retrieve(question)
    completion = llm.chat.completions.create(
        model=CHATBOT_MODEL,
        messages=[
            {
                "role": "user",
                "content": RAG_PROMPT.format(context="\n---\n".join(contexts), question=question),
            }
        ],
        temperature=0,
    )
    return completion.choices[0].message.content.strip(), contexts


## 3. Collect answers *and* contexts

The dataset now has four columns — `retrieved_contexts` is what unlocks the
retrieval-aware metrics.


In [ ]:
records = []
for i, item in enumerate(test_set, 1):
    answer, contexts = rag_chatbot(item["question"])
    records.append(
        {
            "user_input": item["question"],
            "response": answer,
            "retrieved_contexts": contexts,
            "reference": item["reference"],
        }
    )
    print(f"[{i}/{len(test_set)}] {item['question'][:60]}")


In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 80)
df = pd.DataFrame(records)
df


## 4. Upload the dataset and submit the job

Same flow as Stage 1; this time we use the provider's default metric suite
(`ragas_rag_default`): faithfulness, context precision, context recall, and answer
relevancy.


In [ ]:
import json
from datetime import datetime, timezone

import boto3

run_tag = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
dataset_key = f"stage2/{run_tag}/dataset.jsonl"

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    region_name="us-east-1",
)

if S3_BUCKET not in [b["Name"] for b in s3.list_buckets()["Buckets"]]:
    s3.create_bucket(Bucket=S3_BUCKET)

jsonl = "\n".join(json.dumps(r) for r in records)
s3.put_object(Bucket=S3_BUCKET, Key=dataset_key, Body=jsonl.encode())
print(f"Uploaded {len(records)} rows to s3://{S3_BUCKET}/{dataset_key}")


In [ ]:
from evalhub import ModelConfig, SyncEvalHubClient
from evalhub.models.api import (
    BenchmarkConfig,
    ExperimentConfig,
    JobSubmissionRequest,
    ModelAuth,
    S3TestDataRef,
    TestDataRef,
)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_TOKEN,
    tenant=EVALHUB_TENANT,
    timeout=60.0,
)

request = JobSubmissionRequest(
    name=f"rag-stage2-{run_tag}",
    description="Stage 2: retrieval-aware RAG metrics",
    experiment=ExperimentConfig(name=EXPERIMENT_NAME),
    model=ModelConfig(
        url=JUDGE_URL,
        name=JUDGE_MODEL,
        auth=ModelAuth(secret_ref=JUDGE_AUTH_SECRET),
    ),
    benchmarks=[
        BenchmarkConfig(
            id="ragas_rag_default",
            provider_id="ragas",
            parameters={
                "metrics": [
                    "faithfulness",
                    "context_precision",
                    "context_recall",
                    "answer_relevancy",
                ],
                "embedding_model": JUDGE_EMBEDDING_MODEL,
            },
            test_data_ref=TestDataRef(
                s3=S3TestDataRef(bucket=S3_BUCKET, key=dataset_key, secret_ref=S3_SECRET_REF)
            ),
        )
    ],
)

job = client.jobs.submit(request)
print(f"Job submitted: {job.id} (state: {job.state})")


In [ ]:
final = client.jobs.wait_for_completion(job.id, timeout=1800, poll_interval=10)
print(f"Job {final.id} finished: {final.effective_state}")

scores = final.results.benchmarks[0].metrics
summary = pd.DataFrame(
    [
        {"metric": name, "score": round(float(value), 3)}
        for name, value in sorted(scores.items())
    ]
)
summary


## 5. Diagnosis: which knob to turn

Read the scores together — each combination points at a different fix:

| Symptom | Diagnosis | Where to intervene |
|---------|-----------|--------------------|
| Low `faithfulness`, good `context_recall` | The model had the right material and ignored it | **Generation**: prompt ("answer only from context"), model choice, temperature |
| Low `context_recall` | The needed information never reached the model | **Retrieval**: chunking, embeddings, top-k, query rewriting, missing documents |
| Low `context_precision` | Retrieval returns junk alongside the signal | **Retrieval**: better ranking, re-ranking step, smaller top-k |
| Low `answer_relevancy` | Answers are evasive or off-topic | **Generation**: prompt and instruction tuning |
| Everything high, but Stage 1 scores were low | Chatbot is consistent with its sources — the sources themselves are wrong/outdated | **Knowledge base content** |

Experiment ideas with this notebook: lower `top_k` to 1 (watch `context_recall` drop),
add more distractor documents (watch `context_precision` drop), or remove the
"use ONLY the context" instruction (watch `faithfulness` drop).

**Make it a habit.** Re-run both stages before and after every significant change to the
pipeline — retrieval settings, prompts, models, document updates — and track scores over
time. EvalHub keeps the history of every job (`client.jobs.list()`), so regressions are
easy to spot.


## Track your runs in MLflow

Because the job carried an `experiment` config, EvalHub logged this evaluation as an
MLflow run: the aggregate metrics, plus the **per-question scores** as `results.jsonl`
and `results.csv` artifacts (browse them in the MLflow UI — the URL is printed below).

This is what makes evaluation a habit instead of a one-off: every re-run of this
notebook appends a row here. Re-run before and after every significant change —
new documents, prompt tweaks, model upgrades — and a score that drops is a
conversation worth having before the change ships.


In [ ]:
import requests

MLFLOW_API = f"{MLFLOW_URL}/api/2.0/mlflow"

exp = requests.get(
    f"{MLFLOW_API}/experiments/get-by-name", params={"experiment_name": EXPERIMENT_NAME}
).json()["experiment"]
runs = requests.post(
    f"{MLFLOW_API}/runs/search",
    json={"experiment_ids": [exp["experiment_id"]], "max_results": 100},
).json().get("runs", [])

history = pd.DataFrame(
    [
        {
            "run": r["info"].get("run_name", r["info"]["run_id"][:8]),
            "when": pd.to_datetime(int(r["info"]["start_time"]), unit="ms").round("s"),
            **{m["key"]: round(m["value"], 3) for m in r["data"].get("metrics", [])},
        }
        for r in runs
    ]
).sort_values("when").reset_index(drop=True)

print(f"MLflow UI: {MLFLOW_URL}/#/experiments/{exp['experiment_id']}")
history
